# 01 — Scrape Guardian News API
Fetches articles tagged with AI-related keywords from The Guardian API.
Output: `data/raw_articles.csv`

In [9]:
import requests
import pandas as pd
import time
import os

os.makedirs('data', exist_ok=True)

API_KEY = ''
BASE_URL = 'https://content.guardianapis.com/search'

# ── Query ────────────────────────────────────────────────────
# Tighter than before: require exact phrases so "AI" in passing
# (e.g. sports match reports) doesn't trigger a match.
# The Guardian q= parameter does full-body search, so we rely on
# section filtering + post-fetch relevance filtering to clean up.
QUERY = (
    '"artificial intelligence" OR "machine learning" OR '
    '"generative AI" OR "large language model" OR "ChatGPT" OR '
    '"AI regulation" OR "AI safety" OR "AI Act" OR '
    '"deep learning" OR "neural network" OR "OpenAI" OR '
    '"Google DeepMind" OR "AI bias" OR "AI ethics" OR "algorithm" OR "robot" OR "automation" OR '
    '"Anthropic" OR "Gemini" OR "Copilot"'
)

FROM_DATE = '2021-01-01'
TO_DATE   = '2026-02-01'
PAGE_SIZE = 200   # API maximum

# ── Section allow-list ───────────────────────────────────────
# Only fetch from sections where AI coverage is substantive.
# The Guardian API supports `section=` as a comma-separated filter.
# Avoids sports, entertainment, lifestyle pulling in stray matches.
ALLOWED_SECTIONS = [
    'technology', 'business', 'science', 'politics',
    'world', 'us-news', 'uk-news',
    'media', 'environment', 'society', 'global-development',
    'education', 'law', 'money', 'news', 'opinion',
    'australia-news', 'the-filter',
]

# ── Relevance filter (post-fetch) ───────────────────────────
# Require at least one AI term to appear in the headline OR
# the first 300 chars of body — avoids deep-body incidental matches.
RELEVANCE_TERMS = [
    'artificial intelligence', ' ai ', 'ai,', 'machine learning',
    'generative ai', 'large language model', 'llm', 'chatgpt',
    'gpt-4', 'gpt-3', 'openai', 'deep learning', 'neural network',
    'ai regulation', 'ai safety', 'ai act', 'deepmind', 'ai bias',
    'ai ethics', 'algorithm', 'robot', 'automation', 'midjourney',
    'stable diffusion', 'anthropic', 'gemini', 'copilot', 'claude', 'openclaw'
]

def is_ai_relevant(headline: str, body: str) -> bool:
    """Return True if an AI term appears in headline or lead paragraph."""
    text = (str(headline) + ' ' + str(body)[:3000]).lower()
    return any(term in text for term in RELEVANCE_TERMS)

In [10]:
def fetch_page(page: int) -> dict:
    params = {
        'q':          QUERY,
        'from-date':  FROM_DATE,
        'to-date':    TO_DATE,
        'page-size':  PAGE_SIZE,
        'page':       page,
        'show-fields': 'bodyText,headline,wordcount',
        'section':    '|'.join(ALLOWED_SECTIONS),  # server-side section filter
        'api-key':    API_KEY,
    }
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    return r.json()['response']


def parse_results(results: list) -> list:
    rows = []
    for item in results:
        fields = item.get('fields', {})
        headline = fields.get('headline', '')
        body     = fields.get('bodyText', '')

        # Skip articles where AI terms don't appear in headline or lead
        if not is_ai_relevant(headline, body):
            continue

        rows.append({
            'id':        item.get('id'),
            'date':      item.get('webPublicationDate', '')[:10],
            'section':   item.get('sectionName'),
            'headline':  headline,
            'body':      body,
            'wordcount': fields.get('wordcount', 0),
            'url':       item.get('webUrl'),
        })
    return rows

In [11]:
import sys

def fetch_page_with_retry(page: int, retries: int = 3, backoff: float = 5.0) -> dict:
    """Fetch a page with exponential backoff on 5xx errors."""
    for attempt in range(retries):
        try:
            return fetch_page(page)
        except requests.HTTPError as e:
            if e.response.status_code >= 500 and attempt < retries - 1:
                wait = backoff * (2 ** attempt)
                print(f'  [503] Page {page} failed, retrying in {wait:.0f}s...', flush=True)
                time.sleep(wait)
            else:
                raise

# ── Paginate ─────────────────────────────────────────────────
all_rows = []
first = fetch_page_with_retry(1)
total_pages = first['pages']
total_articles = first['total']
print(f'API reports {total_articles:,} articles across {total_pages} pages')

all_rows.extend(parse_results(first['results']))

for page in range(2, total_pages + 1):
    try:
        resp = fetch_page_with_retry(page)
        all_rows.extend(parse_results(resp['results']))
    except Exception as e:
        print(f'  Page {page} failed permanently: {e}', file=sys.stderr)

    if page % 10 == 0 or page == total_pages:
        print(f'  Page {page}/{total_pages} — kept so far: {len(all_rows):,}', flush=True)

    time.sleep(5)

print(f'\nDone. Raw rows after relevance filter: {len(all_rows):,}')

API reports 6,441 articles across 33 pages
  Page 10/33 — kept so far: 1,845
  Page 20/33 — kept so far: 3,232
  Page 30/33 — kept so far: 4,350
  Page 33/33 — kept so far: 4,502

Done. Raw rows after relevance filter: 4,502


In [12]:
df = pd.DataFrame(all_rows)
df['date'] = pd.to_datetime(df['date'])
df.drop_duplicates(subset='id', inplace=True)

print(f'\nFinal dataset: {len(df)} articles')
df.head()


Final dataset: 4496 articles


,id,date,section,headline,body,wordcount,url
0,technology/2025/nov/14/people-in-the-uk-have-y...,2025-11-14,Technology,People in the UK: have you received good or ba...,Tech companies are pumping billions into the g...,151,https://www.theguardian.com/technology/2025/no...
1,technology/2026/jan/28/artificial-intelligence...,2026-01-28,Technology,"Artificial intelligence will cost jobs, admits...",Increasing deployment of artificial intelligen...,470,https://www.theguardian.com/technology/2026/ja...
2,australia-news/2025/nov/12/australian-governme...,2025-11-12,Australia news,Australian government could explore using AI f...,The federal government could “explore” using a...,664,https://www.theguardian.com/australia-news/202...
3,technology/2025/jul/17/ai-firms-unprepared-for...,2025-07-17,Technology,AI firms ‘unprepared’ for dangers of building ...,Artificial intelligence companies are “fundame...,527,https://www.theguardian.com/technology/2025/ju...
4,australia-news/2025/nov/12/government-using-ma...,2025-11-12,Australia news,Government using machine learning to help crea...,National Disability Insurance Agency staff are...,908,https://www.theguardian.com/australia-news/202...


In [13]:
df = pd.DataFrame(all_rows)
df['date'] = pd.to_datetime(df['date'])
df.drop_duplicates(subset='id', inplace=True)

# Quality filter: drop very short or empty articles
df['wordcount'] = pd.to_numeric(df['wordcount'], errors='coerce').fillna(0).astype(int)
df = df[(df['wordcount'] >= 100) & (df['body'].str.strip() != '')]
df = df.reset_index(drop=True)

print(f'Final dataset: {len(df):,} articles')
print(f'\nSection distribution:')
print(df['section'].value_counts().head(20).to_string())
print(f'\nDate range: {df["date"].min().date()} → {df["date"].max().date()}')

df.to_csv('data/raw_articles.csv', index=False)
print('\nSaved: data/raw_articles.csv')

# Delete stale embedding cache — corpus has changed, must re-embed in NB 02
import os
if os.path.exists('data/embeddings.npy'):
    os.remove('data/embeddings.npy')
    print('Deleted stale data/embeddings.npy — will be regenerated in NB 02')

Final dataset: 4,490 articles

Section distribution:
section
Technology            1720
Business               542
US news                358
World news             358
Australia news         344
Science                223
Politics               190
Society                169
Environment            151
Media                  105
UK news                 93
Education               83
News                    64
Money                   43
Global development      36
Law                     11

Date range: 2021-01-01 → 2026-02-01

Saved: data/raw_articles.csv
